# CRNViewer Investigation
This notebook checks two paths:
1. interactive notebook widget via `display_container_widget`
2. standalone HTML export via `cybuilder_html` and `state_html`

In [1]:
from __future__ import annotations

from pathlib import Path
from dataclasses import dataclass

from crnviewer.html import cybuilder_html, state_html
from crnviewer.ipython import (
    build_builder_from_container,
    display_container_widget,
    container_state,
)

In [2]:
@dataclass(frozen=True)
class DemoContainer:
    _species: tuple[str, ...]
    _reactions: tuple[str, ...]
    _mapping: dict[str, tuple[tuple[str, ...], tuple[str, ...]]]

    @property
    def species(self):
        return self._species

    @property
    def reactions(self):
        return self._reactions

    def reactants_and_products_from_reaction(self, reaction: str):
        return self._mapping[reaction]


container = DemoContainer(
    _species=("CH4", "O2", "CO2", "H2O"),
    _reactions=("rxn1",),
    _mapping={
        "rxn1": (("CH4", "O2"), ("CO2", "H2O"))
    },
)

In [3]:
widget = display_container_widget(
    container,
    height="500px",
    width="100%",
)
widget

HTML(value='<iframe style=\'width:100%;height:500px;border:0;\' sandbox=\'allow-scripts allow-downloads\' srcd…

In [4]:
builder = build_builder_from_container(container)
state = container_state(container, layout="layered", layout_options={"elk.direction": "RIGHT"})

out_dir = Path("../investigation_outputs")
out_dir.mkdir(parents=True, exist_ok=True)

(out_dir / "notebook_network_direct.html").write_text(
    cybuilder_html(builder, layout="layered", layout_options={"elk.direction": "RIGHT"}),
    encoding="utf-8",
)
(out_dir / "notebook_network_state.html").write_text(
    state_html(state),
    encoding="utf-8",
)

print("Wrote notebook HTML artifacts:")
print(out_dir / "notebook_network_direct.html")
print(out_dir / "notebook_network_state.html")
print("State keys:", sorted(state.keys()))

Wrote notebook HTML artifacts:
..\investigation_outputs\notebook_network_direct.html
..\investigation_outputs\notebook_network_state.html
State keys: ['elements', 'layout', 'schema_version', 'style', 'ui']


## RDKit container construction

This section investigates constructing CRN containers directly from:
- atom-mapped reaction SMILES
- RDKit `ChemicalReaction` objects

In [5]:
from rdkit.Chem import rdChemReactions

from crnviewer.crn import CRGContainer, SCRGContainer

mapped = "[CH3:1].[OH:2]>>[CH3:1][OH:2]"

scrg_from_smiles = SCRGContainer.from_reaction_smiles(mapped)
crg_from_smiles = CRGContainer.from_reaction_smiles(mapped)

rxn = rdChemReactions.ReactionFromSmarts(mapped, useSmiles=True)
scrg_from_rdkit = SCRGContainer.from_rdkit_reactions([rxn])

print("SCRG (mapped smiles):", len(scrg_from_smiles.reactions), "reaction(s)")
print("CRG  (mapped smiles):", len(crg_from_smiles.reactions), "reaction(s)")
print("SCRG (RDKit rxn):", len(scrg_from_rdkit.reactions), "reaction(s)")

reaction = next(iter(scrg_from_rdkit.reactions))
print("formed bonds:", sorted(tuple(b) for b in reaction.get_formed_bonds()))
print("broken bonds:", sorted(tuple(b) for b in reaction.get_broken_bonds()))

SCRG (mapped smiles): 1 reaction(s)
CRG  (mapped smiles): 1 reaction(s)
SCRG (RDKit rxn): 1 reaction(s)
formed bonds: [(1, 2)]
broken bonds: []


## Zenodo mapped reactions smoke test

Using three atom-mapped reactions sampled from Zenodo record 3715478 (`b97d3.csv`, rows 0, 1, and 3) to validate `SCRGContainer.from_reaction_smiles` in notebook context.

In [6]:
from crnviewer.crn import SCRGContainer

zenodo_mapped_reactions = [
    "[C:1]([c:2]1[n:3][o:4][n:5][n:6]1)([H:7])([H:8])[H:9]>>[C:1]1([H:7])([H:8])/[C:2](=[N:3]\\[H:9])[N:6]1[N:5]=[O:4]",
    "[C:1]([c:2]1[n:3][o:4][n:5][n:6]1)([H:7])([H:8])[H:9]>>[C:1]([C:2](=[N:3][O-:4])[N+:6]#[N:5])([H:7])([H:8])[H:9]",
    "[C:1]([c:2]1[n:3][o:4][n:5][n:6]1)([H:7])([H:8])[H:9]>>[C:1]([C:2]#[N:3])([H:7])([H:8])[H:9].[O:4]=[N+:5]=[N-:6]",
]

for idx, mapped in enumerate(zenodo_mapped_reactions, start=1):
    container = SCRGContainer.from_reaction_smiles(mapped)
    reaction = next(iter(container.reactions))
    print(
        f"sample {idx}: reactions={len(container.reactions)}, "
        f"species={len(container.species)}, atoms={reaction.n_atoms}"
    )

sample 1: reactions=1, species=2, atoms=9
sample 2: reactions=1, species=2, atoms=9
sample 3: reactions=1, species=3, atoms=9


In [7]:
from crnviewer.crn import display_rdkit_mapped_reaction_widget
from crnviewer.rdkit_render import RDKitDrawOptions

display_rdkit_mapped_reaction_widget(
    zenodo_mapped_reactions[0],
    height="500px",
    width="100%",
    render_options=RDKitDrawOptions(includeAtomMaps=False),
    backend="html",
)

[16:09:21] reactant 0 has no mapped atoms.
[16:09:21] product 0 has no mapped atoms.


HTML(value='<iframe style=\'width:100%;height:500px;border:0;\' sandbox=\'allow-scripts allow-downloads\' srcd…

In [8]:
from IPython.display import Markdown, display
from crnviewer.crn import display_rdkit_mapped_reaction_widget
from crnviewer.rdkit_render import RDKitDrawOptions

options = RDKitDrawOptions(includeAtomMaps=False)

for idx, mapped in enumerate(zenodo_mapped_reactions, start=1):
    display(Markdown(f"### Zenodo sample {idx}"))
    display(
        display_rdkit_mapped_reaction_widget(
            mapped,
            height="420px",
            width="100%",
            render_options=options,
            backend="html",
        )
    )

### Zenodo sample 1

[16:09:21] reactant 0 has no mapped atoms.
[16:09:21] product 0 has no mapped atoms.


HTML(value='<iframe style=\'width:100%;height:420px;border:0;\' sandbox=\'allow-scripts allow-downloads\' srcd…

### Zenodo sample 2

[16:09:21] reactant 0 has no mapped atoms.
[16:09:21] product 0 has no mapped atoms.


HTML(value='<iframe style=\'width:100%;height:420px;border:0;\' sandbox=\'allow-scripts allow-downloads\' srcd…

### Zenodo sample 3

[16:09:21] reactant 0 has no mapped atoms.
[16:09:21] product 0 has no mapped atoms.
[16:09:21] product 1 has no mapped atoms.


HTML(value='<iframe style=\'width:100%;height:420px;border:0;\' sandbox=\'allow-scripts allow-downloads\' srcd…

In [9]:
w = display_rdkit_mapped_reaction_widget(
    zenodo_mapped_reactions[0],
    render_options=RDKitDrawOptions(includeAtomMaps=False),
)
print(len(w.cytoscape_js), len(w.elk_js), len(w.cytoscape_elk_js), len(w.klay_js), len(w.cytoscape_klay_js), len(w.cytoscape_cose_bilkent_js))
w

[16:09:22] reactant 0 has no mapped atoms.
[16:09:22] product 0 has no mapped atoms.


AttributeError: 'HTML' object has no attribute 'cytoscape_js'